In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import mean_absolute_error

# ── CONFIG ───────────────────────────────────────────────────
PALETTE  = ['#4C72B0','#DD8452','#55A868','#C44E52','#8172B2','#937860','#DA8BC3']
BG       = '#F8F9FA'
SEED     = 42
N_SAMPLE = 100_000

TARGETS = ['CSN6', 'AGR9', 'AGR5', 'EXT4', 'EST7', 'OPN3', 'OPN10']

Q_COLS = [
    'EXT1','EXT2','EXT3','EXT4','EXT5','EXT6','EXT7','EXT8','EXT9','EXT10',
    'EST1','EST2','EST3','EST4','EST5','EST6','EST7','EST8','EST9','EST10',
    'AGR1','AGR2','AGR3','AGR4','AGR5','AGR6','AGR7','AGR8','AGR9','AGR10',
    'CSN1','CSN2','CSN3','CSN4','CSN5','CSN6','CSN7','CSN8','CSN9','CSN10',
    'OPN1','OPN2','OPN3','OPN4','OPN5','OPN6','OPN7','OPN8','OPN9','OPN10',
]
FEATURES = [c for c in Q_COLS if c not in TARGETS]  # 43 features

TRAIT_MAP = {
    'EXT': 'Extraversion',
    'EST': 'Neuro-stability',
    'AGR': 'Agreeableness',
    'CSN': 'Conscientiousness',
    'OPN': 'Openness',
}

# ─────────────────────────────────────────────────────────────
# SECTION 1 : LOAD & CLEAN
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("  SECTION 1 : LOAD & CLEAN")
print("=" * 60)



FILE_PATH = 'data-final.csv'

df_raw = pd.read_csv(FILE_PATH)
print(f"  Raw shape        : {df_raw.shape}")

# กรองเฉพาะ 50 คำถาม, ค่า 1–5 เท่านั้น (ไม่มี 0 = ไม่ตอบ)
df = df_raw[Q_COLS].copy()
df = df[(df[Q_COLS] >= 1).all(axis=1) & (df[Q_COLS] <= 5).all(axis=1)]
print(f"  After cleaning   : {df.shape}")

# Sample 100k
df = df.sample(N_SAMPLE, random_state=SEED).reset_index(drop=True)
print(f"  After sampling   : {df.shape}")

# ─────────────────────────────────────────────────────────────
# SECTION 2 : EDA
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SECTION 2 : EDA")
print("=" * 60)

# ── 2A. Distribution ของ 7 Targets ──────────────────────────
fig, axes = plt.subplots(1, 7, figsize=(22, 4), facecolor=BG)
fig.suptitle('Distribution of 7 Target Variables', fontsize=14, fontweight='bold', y=1.02)

for ax, col, c in zip(axes, TARGETS, PALETTE):
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=c, edgecolor='white', width=0.7)
    ax.set_title(col, fontsize=13, fontweight='bold')
    ax.set_xlabel('Score (1–5)', fontsize=9)
    ax.set_facecolor(BG)
    ax.spines[['top', 'right']].set_visible(False)
    mean_v = df[col].mean()
    ax.axvline(mean_v, color='red', linestyle='--', lw=1.8, label=f'μ={mean_v:.2f}')
    ax.legend(fontsize=9)

plt.tight_layout()
plt.show()

# ── 2B. Trait-Level Mean Scores ──────────────────────────────
trait_means = {}
for pfx, name in TRAIT_MAP.items():
    cols = [c for c in Q_COLS if c.startswith(pfx)]
    trait_means[name] = df[cols].mean().mean()

fig, ax = plt.subplots(figsize=(9, 4), facecolor=BG)
bars = ax.barh(list(trait_means.keys()), list(trait_means.values()),
               color=PALETTE[:5], edgecolor='white', height=0.6)
ax.set_xlim(1, 5)
ax.axvline(3, color='gray', linestyle='--', lw=1.2, alpha=0.5, label='Midpoint (3)')
ax.set_xlabel('Mean Score', fontsize=11)
ax.set_title('Average Score per Big Five Trait', fontsize=13, fontweight='bold')
ax.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(fontsize=9)
for bar, val in zip(bars, trait_means.values()):
    ax.text(val + 0.05, bar.get_y() + bar.get_height() / 2,
            f'{val:.2f}', va='center', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 2C. Correlation Heatmap (50 คำถาม) ──────────────────────
fig, ax = plt.subplots(figsize=(16, 14), facecolor=BG)
corr = df[Q_COLS].corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdYlBu_r', center=0, vmin=-1, vmax=1,
            linewidths=0.3, ax=ax, annot=False, cbar_kws={'shrink': 0.7})
ax.set_title('Correlation Matrix – All 50 Questions', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 2D. Boxplot ของ Targets ──────────────────────────────────
fig, axes = plt.subplots(1, 7, figsize=(22, 5), facecolor=BG)
fig.suptitle('Score Boxplots – Target Variables', fontsize=13, fontweight='bold', y=1.02)

for ax, col, c in zip(axes, TARGETS, PALETTE):
    data_by_score = [df[df[col] == v][col].values for v in range(1, 6)]
    bp = ax.boxplot(data_by_score, positions=range(1, 6), patch_artist=True,
                    boxprops=dict(facecolor=c, alpha=0.75),
                    medianprops=dict(color='black', lw=2),
                    whiskerprops=dict(lw=1.2),
                    capprops=dict(lw=1.2),
                    flierprops=dict(marker='.', alpha=0.3, markersize=3))
    ax.set_xlabel('Score', fontsize=9)
    ax.set_title(col, fontsize=12, fontweight='bold')
    ax.set_facecolor(BG)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── 2E. Score Distribution ต่อ Trait (violin) ───────────────
fig, axes = plt.subplots(1, 5, figsize=(20, 5), facecolor=BG)
fig.suptitle('Score Distribution per Trait (All Questions)', fontsize=13, fontweight='bold')

for ax, (pfx, name), c in zip(axes, TRAIT_MAP.items(), PALETTE):
    cols = [col for col in Q_COLS if col.startswith(pfx)]
    all_vals = df[cols].values.flatten()
    ax.violinplot(all_vals, positions=[1], showmedians=True)
    ax.set_title(f'{pfx}\n({name})', fontsize=10, fontweight='bold')
    ax.set_xticks([])
    ax.set_ylabel('Score', fontsize=9)
    ax.set_facecolor(BG)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print("  ✓ EDA เสร็จสิ้น")

# ─────────────────────────────────────────────────────────────
# SECTION 3 : FEATURE ENGINEERING
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SECTION 3 : FEATURE ENGINEERING")
print("=" * 60)

df_fe = df[FEATURES].copy()

# A) Trait Aggregate Stats (mean, std, range) ต่อ 5 trait
for pfx in ['EXT', 'EST', 'AGR', 'CSN', 'OPN']:
    cols = [c for c in FEATURES if c.startswith(pfx)]
    if cols:
        df_fe[f'{pfx}_mean']  = df_fe[cols].mean(axis=1)
        df_fe[f'{pfx}_std']   = df_fe[cols].std(axis=1)
        df_fe[f'{pfx}_range'] = df_fe[cols].max(axis=1) - df_fe[cols].min(axis=1)

# B) Cross-Trait Interactions
df_fe['EXT_EST_ratio'] = df_fe['EXT_mean'] / (df_fe['EST_mean'] + 1e-5)
df_fe['AGR_CSN_sum']   = df_fe['AGR_mean'] + df_fe['CSN_mean']
df_fe['OPN_EXT_prod']  = df_fe['OPN_mean'] * df_fe['EXT_mean']
df_fe['total_mean']    = df_fe[[c for c in df_fe.columns if c.endswith('_mean')]].mean(axis=1)

# C) Response Style Features
df_fe['response_var']    = df_fe[FEATURES].var(axis=1)
df_fe['extreme_count']   = ((df_fe[FEATURES] == 1) |
                             (df_fe[FEATURES] == 5)).sum(axis=1)
df_fe['mid_count']       = (df_fe[FEATURES] == 3).sum(axis=1)
df_fe['acquiescence']    = (df_fe[FEATURES] >= 4).sum(axis=1)

print(f"  Features ก่อน engineering : {len(FEATURES)}")
print(f"  Features หลัง engineering : {df_fe.shape[1]}")
new_feats = [c for c in df_fe.columns if c not in FEATURES]
print(f"  Features ใหม่ที่เพิ่ม ({len(new_feats)}) : {new_feats}")

# Visualise Engineered Features
trait_eng_feats = [f'{p}_{s}' for p in ['EXT','EST','AGR','CSN','OPN']
                   for s in ['mean','std','range']]
fig, axes = plt.subplots(3, 5, figsize=(22, 12), facecolor=BG)
fig.suptitle('Feature Engineering – Trait Aggregates', fontsize=13, fontweight='bold')

for ax, feat, c in zip(axes.flat, trait_eng_feats, PALETTE * 3):
    ax.hist(df_fe[feat], bins=30, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(feat, fontsize=10, fontweight='bold')
    ax.set_facecolor(BG)
    ax.spines[['top', 'right']].set_visible(False)
    ax.set_xlabel('Value', fontsize=8)

plt.tight_layout()
plt.show()

# Response Style Features
fig, axes = plt.subplots(1, 4, figsize=(18, 4), facecolor=BG)
fig.suptitle('Response Style Features', fontsize=13, fontweight='bold')
rs_feats = ['response_var', 'extreme_count', 'mid_count', 'acquiescence']
rs_labels = ['Response Variance', 'Extreme Responses (1 or 5)',
             'Midpoint Responses (3)', 'Acquiescence (≥4)']

for ax, feat, label, c in zip(axes, rs_feats, rs_labels, PALETTE):
    ax.hist(df_fe[feat], bins=30, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(label, fontsize=10, fontweight='bold')
    ax.set_facecolor(BG)
    ax.spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

print("  ✓ Feature Engineering เสร็จสิ้น")

# ─────────────────────────────────────────────────────────────
# SECTION 4 : CLUSTERING (KMeans)
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SECTION 4 : CLUSTERING")
print("=" * 60)

TRAIT_COLS = ['EXT_mean', 'EST_mean', 'AGR_mean', 'CSN_mean', 'OPN_mean']
scaler     = StandardScaler()
X_cluster  = scaler.fit_transform(df_fe[TRAIT_COLS])

# ── Elbow Method ─────────────────────────────────────────────
inertias = []
K_RANGE  = range(2, 10)
for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10, max_iter=200)
    km.fit(X_cluster)
    inertias.append(km.inertia_)

fig, axes = plt.subplots(1, 2, figsize=(16, 5), facecolor=BG)

axes[0].plot(K_RANGE, inertias, 'o-', color=PALETTE[0], lw=2.5, markersize=9)
axes[0].axvline(4, color='red', linestyle='--', lw=1.8, label='k=4 (chosen)')
axes[0].set_xlabel('Number of Clusters (k)', fontsize=11)
axes[0].set_ylabel('Inertia', fontsize=11)
axes[0].set_title('Elbow Method for Optimal k', fontsize=13, fontweight='bold')
axes[0].set_facecolor(BG)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].legend(fontsize=10)

# ── Fit KMeans k=4 ───────────────────────────────────────────
K_BEST   = 4
km_final = KMeans(n_clusters=K_BEST, random_state=SEED, n_init=10)
df_fe['cluster'] = km_final.fit_predict(X_cluster)

# ── PCA 2D Visualisation ─────────────────────────────────────
pca   = PCA(n_components=2, random_state=SEED)
X_pca = pca.fit_transform(X_cluster)

# Plot sample 10k จุด เพื่อให้ไม่ช้า
plot_idx = np.random.RandomState(SEED).choice(len(X_pca), 10000, replace=False)
for i, c in enumerate(PALETTE[:K_BEST]):
    mask = df_fe['cluster'].values[plot_idx] == i
    axes[1].scatter(X_pca[plot_idx][mask, 0], X_pca[plot_idx][mask, 1],
                    color=c, alpha=0.3, s=6, label=f'Cluster {i}')

axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%})', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%})', fontsize=11)
axes[1].set_title('KMeans Clusters Visualised (PCA 2D)', fontsize=13, fontweight='bold')
axes[1].legend(markerscale=4, fontsize=10)
axes[1].set_facecolor(BG)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── Cluster Personality Profiles ─────────────────────────────
cluster_profile = df_fe.groupby('cluster')[TRAIT_COLS].mean()
print("\n  Cluster Profiles (trait mean scores):")
print(cluster_profile.round(3).to_string())

fig, ax = plt.subplots(figsize=(12, 5), facecolor=BG)
x = np.arange(len(TRAIT_COLS))
w = 0.2
for i, c in enumerate(PALETTE[:K_BEST]):
    ax.bar(x + i * w, cluster_profile.loc[i].values, w,
           color=c, label=f'Cluster {i}', edgecolor='white')
ax.set_xticks(x + w * 1.5)
ax.set_xticklabels([t.replace('_mean', '') for t in TRAIT_COLS], fontsize=12)
ax.set_ylabel('Mean Score', fontsize=11)
ax.set_title('Personality Profiles per Cluster', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# Distribution ของ cluster
print("\n  Cluster Sizes:")
print(df_fe['cluster'].value_counts().sort_index().to_string())

print("  ✓ Clustering เสร็จสิ้น")

# ─────────────────────────────────────────────────────────────
# SECTION 5 : EXTRATREES + 5-FOLD CV
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SECTION 5 : ExtraTreesRegressor + 5-Fold CV")
print("=" * 60)
print("  (ExtraTreesRegressor เร็วกว่า RandomForest ~5x MAE ต่างกันแค่ ~0.01)")

X = df_fe.copy()           # 43 original + engineered + cluster
Y = df[TARGETS].copy()

kf = KFold(n_splits=5, shuffle=True, random_state=SEED)


ET_PARAMS = dict(
    n_estimators  = 150,
    max_depth     = 12,
    min_samples_leaf = 8,
    max_features  = 'sqrt',
    n_jobs        = -1,
    random_state  = SEED,
    bootstrap     = False,
)

results        = {}
feat_imp_dict  = {}

for i, target in enumerate(TARGETS):
    y         = Y[target]
    fold_maes = []

    for fold_idx, (tr_idx, val_idx) in enumerate(kf.split(X)):
        et = ExtraTreesRegressor(**ET_PARAMS)
        et.fit(X.iloc[tr_idx], y.iloc[tr_idx])
        pred = et.predict(X.iloc[val_idx])
        fold_maes.append(mean_absolute_error(y.iloc[val_idx], pred))

    # Fit full data เพื่อ feature importance
    et_full = ExtraTreesRegressor(**ET_PARAMS)
    et_full.fit(X, y)

    results[target] = {
        'mae_mean' : np.mean(fold_maes),
        'mae_std'  : np.std(fold_maes),
        'folds'    : fold_maes,
    }
    feat_imp_dict[target] = pd.Series(et_full.feature_importances_, index=X.columns)

    print(f"  [{i+1}/7] {target:<6} MAE = {np.mean(fold_maes):.4f} ± {np.std(fold_maes):.4f}")

print("\n  ✓ Training เสร็จสิ้น")

# ─────────────────────────────────────────────────────────────
# SECTION 6 : RESULT VISUALISATION
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  SECTION 6 : RESULT VISUALISATION")
print("=" * 60)

mae_means = [results[t]['mae_mean'] for t in TARGETS]
mae_stds  = [results[t]['mae_std']  for t in TARGETS]

# ── 6A. MAE Bar + Per-Fold Lines ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(18, 6), facecolor=BG)

axes[0].bar(TARGETS, mae_means, color=PALETTE, edgecolor='white', width=0.6,
            yerr=mae_stds, capsize=7, error_kw={'elinewidth': 2})
axes[0].set_ylabel('MAE', fontsize=12)
axes[0].set_title('MAE per Target (5-Fold CV – ExtraTrees)', fontsize=13, fontweight='bold')
axes[0].set_facecolor(BG)
axes[0].spines[['top', 'right']].set_visible(False)
axes[0].set_ylim(0, max(mae_means) * 1.35)
for i, (m, s) in enumerate(zip(mae_means, mae_stds)):
    axes[0].text(i, m + s + 0.01, f'{m:.3f}', ha='center',
                 fontsize=11, fontweight='bold')

for t, c in zip(TARGETS, PALETTE):
    axes[1].plot(range(1, 6), results[t]['folds'], 'o-',
                 color=c, lw=2.5, markersize=9, label=t)
axes[1].set_xlabel('Fold', fontsize=11)
axes[1].set_ylabel('MAE', fontsize=11)
axes[1].set_title('MAE per Fold – All Targets', fontsize=13, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].set_xticks(range(1, 6))
axes[1].set_facecolor(BG)
axes[1].spines[['top', 'right']].set_visible(False)

plt.tight_layout()
plt.show()

# ── 6B. MAE Heatmap (Targets × Folds) ───────────────────────
fold_df = pd.DataFrame(
    {t: results[t]['folds'] for t in TARGETS},
    index=[f'Fold {i}' for i in range(1, 6)]
)
fig, ax = plt.subplots(figsize=(12, 4), facecolor=BG)
sns.heatmap(fold_df.T, annot=True, fmt='.3f', cmap='YlOrRd',
            linewidths=0.8, ax=ax, cbar_kws={'label': 'MAE'})
ax.set_title('MAE Heatmap – Targets × Folds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# ── 6C. Feature Importance Grid (Top 15 ต่อ target) ─────────
fig, axes = plt.subplots(2, 4, figsize=(26, 13), facecolor=BG)
fig.suptitle('Top 15 Feature Importances per Target (ExtraTrees)',
             fontsize=14, fontweight='bold')

for ax, target, c in zip(axes.flat, TARGETS, PALETTE):
    fi = feat_imp_dict[target].nlargest(15).sort_values()
    ax.barh(fi.index, fi.values, color=c, edgecolor='white', alpha=0.85)
    ax.set_title(f'{target}', fontsize=12, fontweight='bold')
    ax.set_xlabel('Importance', fontsize=10)
    ax.set_facecolor(BG)
    ax.spines[['top', 'right']].set_visible(False)

axes.flat[7].set_visible(False)
plt.tight_layout()
plt.show()

# ── 6D. Overall Feature Importance (average across targets) ──
avg_fi = pd.DataFrame(feat_imp_dict).mean(axis=1).nlargest(20).sort_values()

fig, ax = plt.subplots(figsize=(10, 8), facecolor=BG)
colors = [PALETTE[0] if not any(x in f for x in ['_mean','_std','_range','_ratio',
          '_sum','_prod','total','response','extreme','mid','acqui','cluster'])
          else PALETTE[2] for f in avg_fi.index]
ax.barh(avg_fi.index, avg_fi.values, color=colors, edgecolor='white', alpha=0.85)
ax.set_title('Top 20 Features (Average Importance across 7 Targets)\n'
             '🔵 Original  🟢 Engineered', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean Importance', fontsize=11)
ax.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()
plt.show()

# ─────────────────────────────────────────────────────────────
# SECTION 7 : FINAL SUMMARY
# ─────────────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("  FINAL RESULTS SUMMARY")
print("=" * 60)
print(f"\n  {'Target':<8} {'MAE Mean':>10} {'MAE Std':>10}  {'Interpretation'}")
print("  " + "-" * 55)
for t in TARGETS:
    m = results[t]['mae_mean']
    s = results[t]['mae_std']
    interp = "ดี" if m < 0.60 else ("ปานกลาง" if m < 0.75 else "ยากทำนาย")
    print(f"  {t:<8} {m:>10.4f} {s:>10.4f}  {interp}")
print("  " + "-" * 55)
print(f"  {'Overall':<8} {np.mean(mae_means):>10.4f}")
print(f"\n  Scale คะแนน: 1–5  |  MAE ~0.6 = ผิดพลาดเฉลี่ย 0.6 คะแนน")
print(f"  n_sample = {N_SAMPLE:,}  |  model = ExtraTreesRegressor  |  cv = 5-Fold")

  SECTION 1 : LOAD & CLEAN
  Raw shape        : (1015341, 110)
  After cleaning   : (874434, 50)
  After sampling   : (100000, 50)

  SECTION 2 : EDA
  ✓ EDA เสร็จสิ้น

  SECTION 3 : FEATURE ENGINEERING
  Features ก่อน engineering : 43
  Features หลัง engineering : 66
  Features ใหม่ที่เพิ่ม (23) : ['EXT_mean', 'EXT_std', 'EXT_range', 'EST_mean', 'EST_std', 'EST_range', 'AGR_mean', 'AGR_std', 'AGR_range', 'CSN_mean', 'CSN_std', 'CSN_range', 'OPN_mean', 'OPN_std', 'OPN_range', 'EXT_EST_ratio', 'AGR_CSN_sum', 'OPN_EXT_prod', 'total_mean', 'response_var', 'extreme_count', 'mid_count', 'acquiescence']
  ✓ Feature Engineering เสร็จสิ้น

  SECTION 4 : CLUSTERING

  Cluster Profiles (trait mean scores):
         EXT_mean  EST_mean  AGR_mean  CSN_mean  OPN_mean
cluster                                                  
0           2.926     3.028     3.093     2.820     2.743
1           3.227     3.537     3.540     3.426     3.221
2           2.689     3.229     2.973     3.272     3.303
3   